# TER — Fine-tuning BERT sur ANLI — version GitHub propre

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q transformers[torch] datasets evaluate accelerate optuna scikit-learn matplotlib pandas seaborn captum

# Vérification du GPU
import torch
print(f"GPU détecté : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN GPU'}")

In [ ]:
# 1. Imports standards
import os
import gc
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 2. Imports Hugging Face & Évaluation
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

# 3. Reproductibilité et Nettoyage mémoire
set_seed(42) # Pour que tes résultats soient identiques si tu relances
gc.collect()
torch.cuda.empty_cache()

print("✅ Bibliothèques chargées et mémoire GPU nettoyée.")

In [ ]:
# ==============================
# Configuration Générale Sécurisée
# ==============================

SEED = 42
set_seed(SEED)

# Paramètres du modèle
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
NUM_LABELS = 3


# --- HYPERPARAMÈTRES OPTIMAUX (issus de votre recherche précédente) ---
BEST_PARAMS = {
    'learning_rate': 4.184505461935611e-05,
    'per_device_train_batch_size': 32,
    'num_train_epochs': 3,
    'weight_decay': 0.057427370178733506
}

# --- CHEMINS DE SAUVEGARDE (Google Drive) ---
BASE_DIR = os.environ.get("TER_OUTPUT_DIR", ".")
FINAL_OUTPUT_DIR = os.path.join(BASE_DIR, "models", "final_bert_anli_model")
LOGGING_DIR = f"{DRIVE_PATH}/logs_anli"
# Création des répertoires si nécessaire
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

# Optimisation matérielle
USE_FP16 = torch.cuda.is_available()

print(f"🚀 Configuration prête.")
print(f"📍 Le modèle sera sauvegardé sur Drive : {FINAL_OUTPUT_DIR}")
print(f"💻 GPU : {torch.cuda.get_device_name(0) if USE_FP16 else 'NON DÉTECTÉ'}")

## 1. Chargement du tokenizer et du dataset SNLI

In [ ]:
from datasets import load_dataset, concatenate_datasets

# On garde le même tokenizer BERT
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("🚀 Chargement et fusion du dataset ANLI (R1 + R2 + R3)...")
dataset = load_dataset("anli")

# Fusion des 3 rounds pour le Train, la Validation et le Test
train_dataset_full = concatenate_datasets([
    dataset["train_r1"], dataset["train_r2"], dataset["train_r3"]
]).filter(lambda x: x["label"] != -1)

validation_dataset_full = concatenate_datasets([
    dataset["dev_r1"], dataset["dev_r2"], dataset["dev_r3"]
]).filter(lambda x: x["label"] != -1)

test_dataset_full = concatenate_datasets([
    dataset["test_r1"], dataset["test_r2"], dataset["test_r3"]
]).filter(lambda x: x["label"] != -1)

print(f"✅ Train ANLI fusionné : {len(train_dataset_full)} exemples")
print(f"✅ Validation ANLI fusionnée : {len(validation_dataset_full)} exemples")

## 2. Création des datasets pour Optuna et pour l'entraînement final

On sépare clairement deux phases :

- **Phase Optuna** : recherche des hyperparamètres sur environ 55 000 exemples du train SNLI.
- **Phase finale** : entraînement du modèle sur tout le train SNLI avec les meilleurs hyperparamètres trouvés.

In [ ]:
# ==============================
# Préparation des Datasets
# ==============================

print("Mélange des données et préparation finale...")

# On mélange le dataset complet pour garantir une bonne distribution
# (Le shuffle est important même pour l'entraînement final)
final_train_dataset = train_dataset_full.shuffle(seed=SEED)
final_validation_dataset = validation_dataset_full.shuffle(seed=SEED)
final_test_dataset = test_dataset_full # Pas besoin de mélanger le test

print(f"✅ Datasets prêts pour l'entraînement final :")
print(f"   - Entraînement : {len(final_train_dataset)} exemples")
print(f"   - Validation : {len(final_validation_dataset)} exemples")
print(f"   - Test : {len(final_test_dataset)} exemples")

## 3. Tokenisation

In [ ]:
def preprocess_function(examples):
    # Tokenise les paires prémisse / hypothèse pour BERT.
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

# Colonnes textuelles à retirer pour ne laisser que les vecteurs numériques
columns_to_remove = ["premise", "hypothesis"]

print("🚀 Tokenisation des datasets finaux (cela peut prendre 1 ou 2 minutes)...")

tokenized_final_train = final_train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)

tokenized_final_validation = final_validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)

tokenized_final_test = final_test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)

# Conversion au format PyTorch pour le Trainer
tokenized_final_train.set_format("torch")
tokenized_final_validation.set_format("torch")
tokenized_final_test.set_format("torch")

print("✅ Tokenisation terminée et format PyTorch appliqué.")

## 4. Métriques d'évaluation

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels,
    )

    macro_f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro",
    )

    return {
        "accuracy": accuracy["accuracy"],
        "macro_f1": macro_f1["f1"],
    }

In [ ]:
# ==============================
# Configuration de l'Entraînement Final
# ==============================

final_training_args = TrainingArguments(
    output_dir=FINAL_OUTPUT_DIR,        # Sauvegarde directe sur le Google Drive
    eval_strategy="epoch",              # Évaluation à chaque fin d'époque
    save_strategy="epoch",              # Sauvegarde à chaque fin d'époque (Sécurité)
    logging_strategy="epoch",
    logging_dir=LOGGING_DIR,            # Logs sur le Drive
    report_to="none",
    fp16=USE_FP16,                      # Accélération Colab Pro

    # Injection des meilleurs paramètres récupérés
    learning_rate=BEST_PARAMS['learning_rate'],
    per_device_train_batch_size=BEST_PARAMS['per_device_train_batch_size'],
    per_device_eval_batch_size=BEST_PARAMS['per_device_train_batch_size'],
    num_train_epochs=BEST_PARAMS['num_train_epochs'],
    weight_decay=BEST_PARAMS['weight_decay'],

    # Sécurité et Performance
    load_best_model_at_end=True,        # Recharge le meilleur modèle à la fin
    metric_for_best_model="accuracy",   # Basé sur l'accuracy
    seed=SEED,
    data_seed=SEED,
    push_to_hub=False
)

print(f"✅ Configuration d'entraînement final générée avec les paramètres Optuna.")

## 6. Nettoyage mémoire avant l'entraînement final

In [ ]:
# On s'assure que tout ce qui traîne en mémoire est supprimé
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("🧹 Mémoire nettoyée de fond en comble. La voie est libre pour l'entraînement final !")

In [ ]:
# On s'assure d'utiliser le bon nom de variable (celui que tu as défini)
# Si tu as utilisé 'best_params' en minuscules, garde-le ainsi.

final_training_args = TrainingArguments(
    output_dir=FINAL_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    fp16=USE_FP16,

    # On pioche dans les résultats Optuna
    learning_rate=BEST_PARAMS["learning_rate"],
    per_device_train_batch_size=BEST_PARAMS["per_device_train_batch_size"],
    per_device_eval_batch_size=BEST_PARAMS["per_device_train_batch_size"], # On aligne eval sur train
    num_train_epochs=BEST_PARAMS["num_train_epochs"],
    weight_decay= BEST_PARAMS["weight_decay"],

    # Paramètres de sélection du meilleur modèle
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1", # Très bon choix pour le TER
    greater_is_better=True,

    seed=SEED,
    data_seed=SEED,
    push_to_hub=False
)

print("✅ Arguments d'entraînement final configurés.")

## 7. Entraînement final sur tout le train SNLI

Ici, on repart d'un BERT pré-entraîné neuf, puis on l'entraîne sur tout le dataset SNLI avec les meilleurs hyperparamètres trouvés par Optuna.

In [ ]:
# On utilise BEST_PARAMS (en majuscules) tel que défini à la cellule 3
final_training_args = TrainingArguments(
    output_dir=FINAL_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    fp16=USE_FP16,

    # On pioche dans les résultats optimaux
    learning_rate=BEST_PARAMS["learning_rate"],
    per_device_train_batch_size=BEST_PARAMS["per_device_train_batch_size"],
    per_device_eval_batch_size=BEST_PARAMS["per_device_train_batch_size"],
    num_train_epochs=BEST_PARAMS["num_train_epochs"],
    weight_decay=BEST_PARAMS["weight_decay"],

    # Paramètres de sélection du meilleur modèle
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    seed=SEED,
    data_seed=SEED,
    push_to_hub=False
)

print("✅ Arguments d'entraînement final configurés avec succès.")

In [ ]:
# 1. Chargement du modèle BERT "vierge"
print(f"📥 Chargement de {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# 2. Création du Trainer avec tes paramètres optimaux
final_trainer = Trainer(
    model=model,
    args=final_training_args, # Défini à la cellule précédente
    train_dataset=tokenized_final_train,
    eval_dataset=tokenized_final_validation,
    compute_metrics=compute_metrics,
)

# 3. 🚀 L'ENTRAÎNEMENT (L'étape qui prend du temps)
print("🔥 Lancement de l'entraînement final sur ANLI (R1+R2+R3)...")
final_trainer.train()

# 4. 💾 SAUVEGARDE SÉCURISÉE SUR LE DRIVE
# On sauve le modèle ET le tokenizer au même endroit
final_trainer.save_model(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)

print(f"✅ VICTOIRE ! Modèle entraîné et sauvegardé ici : {FINAL_OUTPUT_DIR}")

## 8. Évaluation finale sur le test set SNLI

In [ ]:
# Cellule 12 : Évaluation sur le dataset de test (que le modèle n'a jamais vu)
print("🧐 Évaluation finale sur le dataset TEST...")
test_results = final_trainer.evaluate(tokenized_final_test)

print("\n📊 RÉSULTATS TEST FINAL :")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")

## 10. Courbes d'apprentissage

Cette partie permet de récupérer automatiquement les pertes d'entraînement et de validation depuis l'historique du `Trainer`.

In [ ]:
# 1. Extraction des logs dans un DataFrame
log_history = pd.DataFrame(final_trainer.state.log_history)

# 2. Séparation des logs d'entraînement et d'évaluation
train_logs = log_history[log_history['loss'].notna()]
eval_logs = log_history[log_history['eval_loss'].notna()]

# 3. Visualisation des courbes (Le "must" pour ton TER)
plt.figure(figsize=(12, 5))

# Graphique de la Perte (Loss)
plt.subplot(1, 2, 1)
plt.plot(train_logs['epoch'], train_logs['loss'], label='Entraînement')
plt.plot(eval_logs['epoch'], eval_logs['eval_loss'], label='Validation')
plt.title('Courbe de Perte (Loss)')
plt.xlabel('Époques')
plt.ylabel('Perte')
plt.legend()

# Graphique de l'Accuracy
plt.subplot(1, 2, 2)
plt.plot(eval_logs['epoch'], eval_logs['eval_accuracy'], label='Accuracy Validation', color='green')
plt.title('Évolution de l\'Accuracy')
plt.xlabel('Époques')
plt.ylabel('Précision')
plt.legend()

plt.tight_layout()
plt.show()

# Affichage du tableau brut pour vérification
log_history.head()

In [ ]:
# Courbe de training loss
train_loss_df = log_history.dropna(subset=["loss"])[["epoch", "loss"]]

plt.figure(figsize=(8, 5))
plt.plot(train_loss_df["epoch"], train_loss_df["loss"], marker="o", label="Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Évolution de la training loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Courbe de validation loss
validation_loss_df = log_history.dropna(subset=["eval_loss"])[["epoch", "eval_loss"]]

plt.figure(figsize=(8, 5))
plt.plot(validation_loss_df["epoch"], validation_loss_df["eval_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Évolution de la validation loss")
plt.legend()
plt.grid(True)
plt.show()

## 11. Résumé des résultats à reprendre dans le rapport

In [ ]:
# 1. On récupère les résultats de validation (le meilleur score atteint)
# Note : final_trainer.evaluate() sans argument évalue sur le eval_dataset
print("📊 Récupération des scores finaux...")
validation_results = final_trainer.evaluate()

# 2. On récupère les résultats sur le dataset de test
test_results = final_trainer.evaluate(tokenized_final_test)

# 3. Création du dictionnaire de synthèse
summary_results = {
    "model_name": MODEL_NAME,
    "best_hyperparameters": BEST_PARAMS,
    "validation_results": validation_results,
    "test_results": test_results,
}

# Affichage propre
import json
print("\n✨ RÉSUMÉ FINAL DE L'EXPÉRIMENTATION :")
print(json.dumps(summary_results, indent=4))

# Optionnel : Sauvegarder ce résumé dans un fichier JSON sur le Drive
import json
with open(f"{FINAL_OUTPUT_DIR}/experiment_summary.json", "w") as f:
    json.dump(summary_results, f, indent=4)
print(f"\n✅ Résumé sauvegardé dans experiment_summary.json")

In [ ]:
import json

# On s'assure d'utiliser les noms de variables définis précédemment
results_bert = {
    "model": "BERT-base-uncased",
    "best_hyperparameters": BEST_PARAMS,
    "validation_results": validation_results,
    "test_results": test_results
}

# Définition du chemin complet sur le Drive
json_file_path = os.path.join(FINAL_OUTPUT_DIR, "results_bert.json")

# Sauvegarde locale ET sur le Drive
with open(json_file_path, "w") as f:
    json.dump(results_bert, f, indent=4)

print(f"✅ Fichier de résultats sauvegardé avec succès dans : {json_file_path}")

# Petit aperçu du contenu
print(json.dumps(results_bert, indent=4))